## Import Libraries

In [ ]:
# Import necessary libraries for data handling, optimization, and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, re, random, time, copy
import pickle
from itertools import permutations
from gurobipy import *
from pathlib import Path
from tqdm import tqdm


In [ ]:
RANDOM_SEED = 11   
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:

# ----------------------------
# your helpers (unchanged)
# ----------------------------

def kpg_player_utility(profile, player):
    """
    profile: dict[player -> list of selected items]
    player: int

    Returns u_player(profile) using list membership only.
    """
    my_items = profile[player]

    # direct profit
    util = sum(profits[player][i] for i in my_items)

    # interaction terms where action[0] == player
    # term interactions[(player,j)][i] is earned if i in my_items and i in profile[j]
    for (a0, a1) in players_interaction:
        if a0 != player:
            continue
        other_items = profile[a1]
        # list membership only
        for i in my_items:
            if i in other_items:
                util += interactions[(a0, a1)][i]
    return float(util)


def from_x_to_obj(profile):
    """
    Computes both individual (selfish) and global objective values.

    Parameters:
        selected_items (dict): Dictionary mapping each player to their selected items.

    Returns:
        selfish_obj (dict): Objective value for each individual player.
        global_obj (float): Total objective value across all players.
    """
    selfish_obj = {p: 0.0 for p in players}

    # direct part
    for p in players:
        selfish_obj[p] += sum(profits[p][i] for i in profile[p])

    # interactions: sum over (a0,a1) and items that are in both players' lists
    for (a0, a1) in players_interaction:
        items0 = profile[a0]
        items1 = profile[a1]
        for i in items0:
            if i in items1:
                selfish_obj[a0] += interactions[(a0, a1)][i]

    global_obj = sum(selfish_obj[p] for p in players)
    return selfish_obj, float(global_obj)



def selfish_KPG(player_fixed):
    """
    MILP best-response for player_fixed:
    max  sum_i profits[player_fixed][i] x_i + sum_{j!=i} sum_{k in S_j} interactions[(i,j)][k] x_k
    s.t. sum_i weights[player_fixed][i] x_i <= budgets[player_fixed]
         x_i in {0,1}
    """
    m = Model(f"KPG_BR_{player_fixed}")
    m.Params.LogToConsole = 0
    m.Params.OutputFlag = 0
    m.Params.Threads = user_Threads
    m.Params.MIPGap = 1e-6

    x = {i: m.addVar(vtype=GRB.BINARY, name=f"x[{i}]") for i in items}
    m.addConstr(quicksum(weights[player_fixed][i] * x[i] for i in items) <= budgets[player_fixed],
                name="Budget")

    # store handles
    m._x = x
    m._player = player_fixed

    # placeholder objective (we will overwrite each solve)
    m.setObjective(0.0, GRB.MAXIMIZE)
    m.update()
    return m


def update_kpg_br_objective(m, profile):
    """
    Update BR objective of model m (for fixed player i) given current profile of others.
    Uses list membership only.
    """
    i = m._player
    x = m._x

    obj = quicksum(profits[i][k] * x[k] for k in items)

    # add interaction coefficients based on others' selected items
    for (a0, a1) in players_interaction:
        if a0 != i:
            continue
        # if item k is selected by player a1, then interaction contributes interactions[(i,a1)][k] * x[k]
        other_items = profile[a1]
        for k in other_items:
            obj += interactions[(a0, a1)][k] * x[k]

    m.setObjective(obj, GRB.MAXIMIZE)
    m.update()



# def solve_br_kpg_linear(m, profile, current_items):
#     update_br_objectives_kpg(m, profile, set(current_items))
#     m.optimize()
#     chosen = [k for k in items if m._x[k].X > 0.5]
#     obj = float(m.ObjVal)  # multi-objective returns primary objective value as ObjVal
#     return chosen, obj


In [ ]:
import random

# ---- helpers ----
def _clip(x, lo, hi):
    return max(lo, min(hi, x))

# p_i = clip(B_i / W_i, eps, 1 - eps)
def _pick_p_i(player, budgets, items, weights, eps=1e-3):
    B_i = float(budgets[player])
    W_i = sum(float(weights[player][j]) for j in items) + 1e-12
    return _clip(B_i / W_i, eps, 1.0 - eps)

# ---- full-support Bernoulli + rejection (no margin) ----
def generate_feasible_solution_KPG_full_support(budgets, items, weights, eps=1e-3, max_rejections=None):
    """
    For each player i:
      p_i = clip(B_i / sum_j w_ij, eps, 1-eps)
      Draw x^i_j ~ Ber(p_i) independently; accept if sum_j w_ij x^i_j <= B_i else resample.
    """
    x_current = {}
    for player in players:
        B_i = float(budgets[player])
        p_i = _pick_p_i(player, budgets, items, weights, eps=eps)

        trials = 0
        while True:
            trials += 1
            chosen = [j for j in items if random.random() < p_i]
            total_w = sum(float(weights[player][j]) for j in chosen)
            if total_w <= B_i:
                x_current[player] = chosen
                break
            if max_rejections is not None and trials >= max_rejections:
                x_current[player] = []  # safe fallback (debug)
                break
    return x_current


def generate_feasible_solution_KPG_maximal(budgets, items, weights): 
    x_current = {} 
    for player in players: 
        selected_items = set() 
        remaining_budget = budgets[player] 
        shuffled = items[:] 
        random.shuffle(shuffled) 
        # random rotate and direction 
        k = random.randrange(len(shuffled)) if shuffled else 0 
        if random.random() < 0.5: 
            seq = shuffled[k:] + shuffled[:k] 
        else: 
            seq = list(reversed(shuffled[k:] + shuffled[:k])) 
        for item in seq: 
            w = weights[player][item] 
            if w <= remaining_budget: 
                selected_items.add(item) 
                remaining_budget -= w 
        x_current[player] = list(selected_items) 
        
    return x_current



# ---- alternating wrapper ----
def generate_feasible_solution_KPG_alternating(trial_index, budgets, items, weights,
                                               eps=1e-3, max_rejections=None,
                                               use_max_on_odd=True):
    """
    Alternate per trial:
      - odd  -> maximal (default)
      - even -> full-support (no margin)
    """
    odd = (trial_index % 2 == 1)
    if (odd and use_max_on_odd) or ((not odd) and (not use_max_on_odd)):
        return generate_feasible_solution_KPG_maximal(budgets, items, weights)
    else:
        return generate_feasible_solution_KPG_full_support(budgets, items, weights,
                                                          eps=eps, max_rejections=max_rejections)


In [ ]:
# ----------------------------
# BRD with end-of-round PNE test ONLY
#  + random permutation EACH round (RR-BRD)
# ----------------------------
def RRR_BRD(x_current, selfish_model, max_rounds):
    # allow None to start fresh model cache per run
    if selfish_model is None:
        selfish_model = {}

    # snapshot initial selection & utilities
    selected_items = {0: {player: [i for i in x_current[player]] for player in players}}
    solution = {0: from_x_to_obj(x_current)[0]}

    TOL = 1e-8
    total_iterations = 0

    # R rounds
    for num in range(1, max_rounds + 1):
        total_iterations += 1

        # --- random permutation THIS round (RR step) ---
        playing_sequence = players[:]           # copy
        random.shuffle(playing_sequence)        # new permutation per round

        # carry over previous round's choices
        selected_items[num] = {player: selected_items[num-1][player][:] for player in players}
        solution[num] = {player: 0 for player in players}

        for player in playing_sequence:
            # Reuse or initialize best-response model for this player
            if player not in selfish_model:
                selfish_model[player] = selfish_KPG(player)
            m = selfish_model[player]

            # incumbent utility under current profile
            u_inc = float(kpg_player_utility(selected_items[num], player))

            # update BR objective using others' current selections
            update_kpg_br_objective(m, selected_items[num])

            # warm start (list membership only)
            incumbent_list = selected_items[num][player]
            for k in items:
                m._x[k].start = 1 if k in incumbent_list else 0

            m.optimize()
            u_br = float(m.ObjVal)

            if u_br > u_inc + TOL:
                selected_items[num][player] = [k for k in items if m._x[k].X > 0.5]
                solution[num][player] = u_br
            else:
                selected_items[num][player] = incumbent_list
                solution[num][player] = u_inc


        # -------- end-of-round PNE test ONLY --------
        # success iff the full round produced no change
        if selected_items[num] == selected_items[num - 1]:
            return selected_items[num], True, total_iterations, solution[num]

    # no PNE within R rounds
    return {player: [] for player in players}, False, total_iterations, {player: 0 for player in players}


In [ ]:
# ----------------------------
# q_mu(R) estimator
# ----------------------------
def estimate_q_mu(R, num_trials=100, seed=None):
    """
    Runs RR-BRD from num_trials random feasible initial profiles.
    PNE is checked only at end-of-round; success if found within <= R rounds.
    Returns success rate q_mu(R).
    """
    if seed is not None:
        random.seed(seed)

    successes = 0
    failures = 0

    # ---- example usage in your loop ----
    for t in range(1, NUM_TRIALS + 1):
        x0 = generate_feasible_solution_KPG_alternating(t, budgets, items, weights, eps=1e-3)
        # print(x0)
        # fresh model cache per run to avoid contamination
        _, found, _, _ = RRR_BRD(x0, selfish_model=None, max_rounds=R)
        if found:
            successes += 1
            print("success")
        else:
            failures += 1
            print("failure")

    denom = successes + failures
    return (successes / denom) if denom > 0 else 0.0, successes, failures

In [ ]:
def list_instances(root_dir: str):
    """Return sorted list of instance file paths matching the spec."""
    hits = []
    for root, _, files in os.walk(root_dir):
        for fname in files:
            if INST_RE.match(fname):
                hits.append(Path(root) / fname)
    # Sort by (n, m, bg, type)
    def sort_key(p: Path):
        m = INST_RE.match(p.name)
        return (int(m.group("n")), int(m.group("m")), int(m.group("bg")), m.group("typ"))
    return sorted(hits, key=sort_key)

def load_instance_into_globals(fpath: Path):
    """
    Loads a single instance file and populates the SAME global variables your functions expect:
      players, items, players_complement, budgets, profits, weights,
      players_interaction, interactions
    """
    global players, items, players_complement, budgets, profits, weights, players_interaction, interactions

    with open(fpath, 'r') as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    num_players, num_items = map(int, lines[0].split())
    items = list(range(num_items))
    players = list(range(num_players))
    players_complement = {p: [q for q in players if q != p] for p in players}
    budgets = list(map(float, lines[1].split()))

    profits = {p: {} for p in players}
    weights = {p: {} for p in players}

    # ordered pairs for interactions (uses itertools.permutations in your scope)
    players_interaction = list(permutations(players, 2))
    interactions = {act: {} for act in players_interaction}

    for ln in lines[2:]:
        data = list(map(int, ln.split()))
        item = data[0]
        # per player profit/weight
        for p in players:
            profits[p][item] = data[2 * p + 1]
            weights[p][item] = data[2 * p + 2]
        # interactions
        offset = 2 * len(players) + 1
        for idx, act in enumerate(players_interaction):
            interactions[act][item] = data[offset + idx]


In [ ]:
def run_q_mu_for_file(fpath: Path, R: int, trials: int, seed: int = None):
    """
    Uses your generate_feasible_solution(...) and RRR_BRD(...) directly.
    Returns:
        q_mu, successes, failures, avg_rounds_all, elapsed_seconds
    where avg_rounds_all = (sum of rounds used over ALL trials) / trials,
    counting failures as R (because RRR_BRD returns total_iterations = R on failure).
    """
    if seed is not None:
        random.seed(seed)

    # load instance into the global symbols your functions expect
    load_instance_into_globals(fpath)

    successes = 0
    failures = 0
    sum_rounds = 0  # sum of total_iterations returned by BRD for every trial

    t0 = time.time()
    # ---- example usage in your loop ----
    for t in range(1, NUM_TRIALS + 1):
        x0 = generate_feasible_solution_KPG_alternating(t, budgets, items, weights, eps=1e-3)
        # Fresh model cache per run to avoid contamination
        _, found, rounds_used, _ = RRR_BRD(x0, selfish_model=None, max_rounds=R)
        sum_rounds += rounds_used
        if found:
            successes += 1
        else:
            failures += 1
    elapsed = time.time() - t0

    denom = successes + failures
    q_mu = (successes / denom) if denom > 0 else 0.0
    avg_rounds_all = (sum_rounds / denom) if denom > 0 else 0.0  # if trials=100, this is exactly "sum/100"
    return q_mu, successes, failures, avg_rounds_all, elapsed


In [ ]:
def summarize_and_save(rows, root_dir: str, tag: str):
    import pandas as pd
    from pathlib import Path

    df = pd.DataFrame(rows).sort_values(["n","m","bg","type"]).reset_index(drop=True)

    # ---- core metrics ----
    num_instances = len(df)
    num_positive = int((df["q_mu(R)"] > 0).sum())
    avg_success = float(df["q_mu(R)"].mean()) if num_instances > 0 else 0.0
    avg_rounds_over_instances = float(df["avg_rounds_all"].mean()) if num_instances > 0 else float("nan")
    # Average ESB over instances with finite values (q>0)
    avg_esb = float(df.loc[df["q_mu(R)"] > 0, "expected_step_bound"].mean()) if num_instances > 0 else float("nan")

    print(f"\n=== Per-instance success rates ({tag}) ===")
    print(df.to_string(index=False, formatters={
        "q_mu(R)": "{:.4f}".format,
        "avg_rounds_all": "{:.2f}".format,
        "expected_step_bound": (lambda x: "inf" if pd.isna(x) else f"{x:.2f}")
    }))

    print(f"\n>>> Summary for {tag}")
    print(f"# instances: {num_instances}")
    print(f"# with q_mu(R) > 0: {num_positive}/{num_instances}")
    print(f"Average success rate over instances: {avg_success:.4f}")
    print(f"Average rounds (all trials, failures counted as R): {avg_rounds_over_instances:.2f}")
    print(f"Average expected step bound (finite only): {avg_esb:.2f}")

    # save main CSV
    out_dir = Path(root_dir)
    out_csv = out_dir / f"q_mu_summary_{tag}_R{R}_trials{NUM_TRIALS}.csv"
    df.to_csv(out_csv, index=False)
    print(f"Saved CSV: {out_csv}")

    # ---- per-type splits (pot, cij, cij-n) ----
    types = ["pot", "cij", "cij-n"]
    present_types = [t for t in types if t in df["type"].unique()]

    per_type_files = []
    for t in present_types:
        dft = df[df["type"] == t].sort_values(["n","m","bg","type"]).reset_index(drop=True)
        path_t = out_dir / f"q_mu_summary_{tag}_{t}_R{R}_trials{NUM_TRIALS}.csv"
        dft.to_csv(path_t, index=False)
        per_type_files.append((t, path_t))

    if per_type_files:
        print("\nSaved per-type CSVs:")
        for t, p in per_type_files:
            print(f"  - {t}: {p}")

    # ---- per-type summary (now includes ESB) ----
    summary_by_type = (
        df.groupby("type", as_index=False)
          .agg(
              num_instances=("filename", "count"),
              num_with_q_pos=("q_mu(R)", lambda s: (s > 0).sum()),
              avg_q_mu_Rm1=("q_mu(R)", "mean"),
              avg_rounds_all_over_instances=("avg_rounds_all", "mean"),
              avg_expected_step_bound=("expected_step_bound", "mean")  # ignores NaN (i.e., q==0)
          )
          .sort_values("type")
          .reset_index(drop=True)
    )
    summary_by_type_csv = out_dir / f"kpg_summary_by_type_{tag}_R{R}_trials{NUM_TRIALS}.csv"
    summary_by_type.to_csv(summary_by_type_csv, index=False)

    print(f"\n=== Summary by type ({tag}) ===")
    print(summary_by_type.to_string(index=False, formatters={
        "avg_q_mu_Rm1": "{:.4f}".format,
        "avg_rounds_all_over_instances": "{:.2f}".format,
        "avg_expected_step_bound": "{:.2f}".format
    }))
    print(f"Saved per-type summary CSV: {summary_by_type_csv}")

    # ---- quick aggregates you already had ----
    for col in ["type","n","m","bg"]:
        avg_col = df.groupby(col)["q_mu(R)"].mean().reset_index()
        print(f"\n=== Averages by {col} ({tag}) ===")
        print(avg_col.to_string(index=False, formatters={"q_mu(R)": "{:.4f}".format}))


In [ ]:
# Run helper
def run_group(files, tag):
    rows = []
    iterator = tqdm(files, desc=f"{tag} instances", unit="inst") if files else []
    for fpath in iterator:
        m = INST_RE.match(fpath.name)
        n = int(m.group("n")); m_items = int(m.group("m")); bg = int(m.group("bg")); typ = m.group("typ")
        q, s, fl, avgR_all, secs = run_q_mu_for_file(fpath, R=R, trials=NUM_TRIALS, seed=RANDOM_SEED)

        # Expected step bound: n*R / q (leave NaN if q==0)
        esb = (n * R / q ) if q > 0 else float("nan")

        rows.append({
            "filename": fpath.name,
            "n": n, "m": m_items, "bg": bg, "type": typ,
            "R": R, "trials": NUM_TRIALS,
            "successes": s, "failures": fl,
            "q_mu(R)": q,              # using the measured success rate column
            "avg_rounds_all": avgR_all,
            "expected_step_bound": esb,  # <-- NEW
            "seconds": round(secs, 2)
        })
    if rows:
        summarize_and_save(rows, ROOT_DIR, tag)
    else:
        print(f"No files in {tag}.")


In [ ]:
ROOT_DIR = r"C:\Users\hyunwoolee\OneDrive - Virginia Tech\Hyunwoo Research\upload\Simulation\generated"
R = 20                  # we will report q_mu(R)
NUM_TRIALS = 200
user_Threads = 16

# file name pattern: n-m-bg-type.txt
INST_RE = re.compile(r"^(?P<n>[23])-(?P<m>25|50|75|100)-(?P<bg>2|5|8)-(?P<typ>pot|cij|cij-n)\.txt$")

# ----------------------------
# Your Set 1 list (base names; .txt optional)
# ----------------------------
SET_INFEAS_BASENAMES = [
    "2-25-2-cij-n",
    "2-25-5-cij-n",
    "2-50-5-cij-n",
    "2-75-5-cij-n",
    "3-25-5-cij-n",
    "3-25-8-cij-n",    
    "3-50-2-cij-n",
    "3-50-5-cij-n",
    "3-50-8-cij-n",
    "3-75-5-cij-n",
    "3-75-8-cij-n",
    "3-100-8-cij-n",
]

SET_TL_BASENAMES = [
    "2-100-5-cij-n",
    "2-100-8-cij-n",
    "3-75-2-cij",
    "3-75-2-cij-n",
    "3-100-2-pot",
    "3-100-2-cij",
    "3-100-5-cij",
    "3-100-8-cij",
    "3-100-2-cij-n",
    "3-100-5-cij-n",
]
# normalize to include .txt for matching
SET_INFEAS_CANON = {f if f.endswith(".txt") else f + ".txt" for f in SET_INFEAS_BASENAMES}
SET_TL_CANON = {f if f.endswith(".txt") else f + ".txt" for f in SET_TL_BASENAMES}


In [ ]:
all_files = list_instances(ROOT_DIR)

# Various Sets
set_infeas_files = [p for p in all_files if p.name in SET_INFEAS_CANON]
set_tl_files = [p for p in all_files if p.name in SET_TL_CANON]
set_pne_files = [p for p in all_files if (p.name not in SET_INFEAS_CANON) and (p.name not in SET_TL_CANON)]

In [ ]:
# Execute
t0 = time.time()
run_group(set_pne_files, "Set_pne")
print(f"\nTotal wall time: {time.time() - t0:.1f}s "
      f"({len(set_infeas_files)} in Set_infeas, {len(set_tl_files)} in Set_tl, {len(set_pne_files)} in Set_pne)")